# B4 · LLM en pratique : Hugging Face, prompting, RAG et LoRA — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/b4_llm/03_notebook.ipynb)

La partie 0 vérifie tes réponses courtes aux quiz, aux rappels et aux exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wb.check`, ou des tests). « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Les questions « dans tes notes » qui n'ont pas de cellule 📝 se notent dans la section « Notes sur le notebook » de ta copie de `06_mes_reponses.md`. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/b4_llm/03_notebook.ipynb`, créée par `python tools/start_chapter.py B4`) : ce fichier-ci est mis à jour par Claude. Sur Colab, le badge ouvre cette version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | B4.Q1 à B4.Q12, B4.R1 à B4.R3, B4.1 à B4.5 | vérifier tes réponses courtes | 🧠 🔁 ✏️ | ★ à ★★ | |
| A | B4.12 à B4.16 | usage et prompting : premier contact avec transformers, générations à trois températures, chat template et streaming, extraction en JSON validé, sensibilité au prompt (*prochaine session de génération*) | | | |
| B | B4.17 à B4.24 | recherche et RAG : chunk_text et BM25, cosine_top_k, RRF et métriques de ta librairie, BM25 contre recherche dense, RAG avec citations, quatre défauts, évaluation, injection (*prochaine session de génération*) | | | |
| C | B4.25 à B4.28 | fine-tuning : LoRA à la main contre peft, style de Holmes, rang et modules ciblés, classification de textes (*prochaine session de génération*) | | | |
| D | B4.29 et B4.30 | livraison : pipeline reproductible et défi RAG (*prochaine session de génération*) | | | |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="B4")  # your own library

## Objectifs et rappel express

- Estimer la mémoire d'un LLM (poids selon le dtype, KV cache) et appliquer à la main son chat template.
- Charger et faire générer de petits modèles avec transformers, écrire et évaluer des prompts sur plusieurs formulations et graines.
- Implémenter le découpage, BM25, la recherche dense, la fusion RRF, recall@k et MRR, puis construire, évaluer et diagnostiquer un RAG avec citations.
- Compter, écrire à la main et appliquer avec peft une adaptation LoRA, et comparer fine-tuning, prompting et classifieur classique.

**Rappel express.** Mémoire des poids $\approx N \times$ octets par paramètre (4 en float32, 2 en bfloat16, $\approx 0{,}5$ en int4) ; KV cache $= 2 \times L \times T \times n_{kv} d_h \times$ octets par valeur. BM25 : $\sum_{t \in q} \operatorname{idf}(t)\,\frac{f_{t,d}(k_1 + 1)}{f_{t,d} + k_1(1 - b + b\,|d| / \overline{|d|})}$, $\operatorname{idf}(t) = \ln\big(1 + \frac{N - n_t + 0{,}5}{n_t + 0{,}5}\big)$. RRF : $\sum_r 1/(k + \operatorname{rang}_r(d))$, $k = 60$. recall@$k$ $= |\text{top-}k \cap \text{pertinents}| / |\text{pertinents}|$ ; MRR = moyenne de $1/\operatorname{rang}$ du premier pertinent. LoRA : $\mathbf{W}' = \mathbf{W} + \frac{\alpha}{r}\mathbf{B}\mathbf{A}$, $\mathbf{B}_0 = 0$, $r(d + k)$ paramètres au lieu de $dk$.

## Partie 0 · Vérifier tes réponses courtes (quiz, rappels, ✏️ B4.1 à B4.5)

Fais d'abord les quiz, les rappels et les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse courte ici : **la valeur** que tu as trouvée, pas l'expression Python, sinon tu ne vérifies rien. Un nombre : `42` ou `0.3` (en Python, le séparateur décimal est un **point** ; `0,3` sans guillemets serait un couple de deux nombres) ; un vrai ou faux : `True` ou `False` ; un choix : la lettre seule, entre guillemets (`"E"`) ; un classement : une liste d'identifiants, du meilleur au moins bon (`[5, 6, 4]`). Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les questions « dans ta copie », la dérivation ∂ B4.6, l'estimation 🧮 B4.7, la réflexion (🗣️ B4.8, ⚖️ B4.9 et B4.10, 📄 B4.11) et l'entretien se corrigent avec `05_solutions.md`.

**B4.Q1 — Modèle de base, modèle instruct, modèle de chat : lequel pour quoi ?**

In [ ]:
# B4.Q1: the values you found on paper
answer_B4_Q1a = ...  # a) the letter of your choice, e.g. "E"
answer_B4_Q1b = ...  # b) True or False
answer_B4_Q1c = ...  # c) the letter of your choice, e.g. "E"
answer_B4_Q1d = ...  # d) the letter of your choice, e.g. "E"
answer_B4_Q1e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B4_Q1a, answer_B4_Q1b, answer_B4_Q1c, answer_B4_Q1d, answer_B4_Q1e]):
    wb.check(f"B4.Q1{letter}", answer)

**B4.Q2 — « Open weights » n'est pas « open source » : lire la licence d'un modèle**

In [ ]:
# B4.Q2: the values you found on paper
answer_B4_Q2a = ...  # a) True or False
answer_B4_Q2b = ...  # b) the letter of your choice, e.g. "E"
answer_B4_Q2c = ...  # c) True or False
answer_B4_Q2d = ...  # d) the letter of your choice, e.g. "E"
answer_B4_Q2e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_B4_Q2a, answer_B4_Q2b, answer_B4_Q2c, answer_B4_Q2d, answer_B4_Q2e]):
    wb.check(f"B4.Q2{letter}", answer)

**B4.Q3 — dtype, device et mémoire : ce qui décide si un modèle tient sur ton ordinateur**

In [ ]:
# B4.Q3: the values you found on paper
answer_B4_Q3a = ...  # a) a number, 1 decimal
answer_B4_Q3b = ...  # b) a number, 1 decimal
answer_B4_Q3c = ...  # c) True or False
answer_B4_Q3d = ...  # d) the letter of your choice, e.g. "E"
answer_B4_Q3e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B4_Q3a, answer_B4_Q3b, answer_B4_Q3c, answer_B4_Q3d, answer_B4_Q3e]):
    wb.check(f"B4.Q3{letter}", answer)

**B4.Q4 — Chat template : pourquoi ne pas écrire soi-même « User: … Assistant: … »**

In [ ]:
# B4.Q4: the values you found on paper
answer_B4_Q4a = ...  # a) the letter of your choice, e.g. "E"
answer_B4_Q4b = ...  # b) True or False
answer_B4_Q4c = ...  # c) the letter of your choice, e.g. "E"
answer_B4_Q4d = ...  # d) True or False
answer_B4_Q4e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_B4_Q4a, answer_B4_Q4b, answer_B4_Q4c, answer_B4_Q4d, answer_B4_Q4e]):
    wb.check(f"B4.Q4{letter}", answer)

**B4.Q5 — Glouton, température, top-p : lequel pour extraire une date, lequel pour écrire un poème ?**

In [ ]:
# B4.Q5: the values you found on paper
answer_B4_Q5a = ...  # a) the letter of your choice, e.g. "E"
answer_B4_Q5b = ...  # b) the letter of your choice, e.g. "E"
answer_B4_Q5c = ...  # c) True or False
answer_B4_Q5d = ...  # d) True or False
answer_B4_Q5e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_B4_Q5a, answer_B4_Q5b, answer_B4_Q5c, answer_B4_Q5d, answer_B4_Q5e]):
    wb.check(f"B4.Q5{letter}", answer)

**B4.Q6 — Zéro-shot, few-shot, chain-of-thought : vrai ou faux**

In [ ]:
# B4.Q6: the values you found on paper
answer_B4_Q6a = ...  # a) True or False
answer_B4_Q6b = ...  # b) True or False
answer_B4_Q6c = ...  # c) True or False
answer_B4_Q6d = ...  # d) True or False
answer_B4_Q6e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B4_Q6a, answer_B4_Q6b, answer_B4_Q6c, answer_B4_Q6d, answer_B4_Q6e]):
    wb.check(f"B4.Q6{letter}", answer)

**B4.Q7 — Pourquoi un LLM invente : hallucination et token le plus probable**

In [ ]:
# B4.Q7: the values you found on paper
answer_B4_Q7a = ...  # a) the letter of your choice, e.g. "E"
answer_B4_Q7b = ...  # b) True or False
answer_B4_Q7c = ...  # c) the letter of your choice, e.g. "E"
answer_B4_Q7d = ...  # d) True or False
answer_B4_Q7e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B4_Q7a, answer_B4_Q7b, answer_B4_Q7c, answer_B4_Q7d, answer_B4_Q7e]):
    wb.check(f"B4.Q7{letter}", answer)

**B4.Q8 — Évaluer un LLM : exact match, juge LLM, contamination des benchmarks**

In [ ]:
# B4.Q8: the values you found on paper
answer_B4_Q8a = ...  # a) a whole number
answer_B4_Q8b = ...  # b) a whole number
answer_B4_Q8c = ...  # c) a number, 3 decimals
answer_B4_Q8d = ...  # d) the letter of your choice, e.g. "E"
answer_B4_Q8e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B4_Q8a, answer_B4_Q8b, answer_B4_Q8c, answer_B4_Q8d, answer_B4_Q8e]):
    wb.check(f"B4.Q8{letter}", answer)

**B4.Q9 — BM25 ou recherche dense : qui retrouve « Watson » et qui retrouve « le docteur » ?**

In [ ]:
# B4.Q9: the values you found on paper
answer_B4_Q9a = ...  # a) the letter of your choice, e.g. "E"
answer_B4_Q9b = ...  # b) True or False
answer_B4_Q9c = ...  # c) the letter of your choice, e.g. "E"
answer_B4_Q9d = ...  # d) the letter of your choice, e.g. "E"
answer_B4_Q9e = ...  # e) a number, 4 decimals

for letter, answer in zip("abcde", [answer_B4_Q9a, answer_B4_Q9b, answer_B4_Q9c, answer_B4_Q9d, answer_B4_Q9e]):
    wb.check(f"B4.Q9{letter}", answer)

**B4.Q10 — RAG ou fine-tuning pour ajouter des connaissances ?**

In [ ]:
# B4.Q10: the values you found on paper
answer_B4_Q10a = ...  # a) the letter of your choice, e.g. "E"
answer_B4_Q10b = ...  # b) the letter of your choice, e.g. "E"
answer_B4_Q10c = ...  # c) True or False
answer_B4_Q10d = ...  # d) True or False
answer_B4_Q10e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_B4_Q10a, answer_B4_Q10b, answer_B4_Q10c, answer_B4_Q10d, answer_B4_Q10e]):
    wb.check(f"B4.Q10{letter}", answer)

**B4.Q11 — LoRA : ce qui est gelé, ce qui est appris, ce que l'on fusionne**

In [ ]:
# B4.Q11: the values you found on paper
answer_B4_Q11a = ...  # a) the letter of your choice, e.g. "E"
answer_B4_Q11b = ...  # b) True or False
answer_B4_Q11c = ...  # c) a whole number
answer_B4_Q11d = ...  # d) the letter of your choice, e.g. "E"
answer_B4_Q11e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B4_Q11a, answer_B4_Q11b, answer_B4_Q11c, answer_B4_Q11d, answer_B4_Q11e]):
    wb.check(f"B4.Q11{letter}", answer)

**B4.Q12 — Injection de prompt : quand un document récupéré devient une instruction**

In [ ]:
# B4.Q12: the values you found on paper
answer_B4_Q12a = ...  # a) the letter of your choice, e.g. "E"
answer_B4_Q12b = ...  # b) True or False
answer_B4_Q12c = ...  # c) the letter of your choice, e.g. "E"
answer_B4_Q12d = ...  # d) True or False
answer_B4_Q12e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_B4_Q12a, answer_B4_Q12b, answer_B4_Q12c, answer_B4_Q12d, answer_B4_Q12e]):
    wb.check(f"B4.Q12{letter}", answer)

**B4.R1 — Ce que le masque causal et le KV cache imposent à la génération**

In [ ]:
# B4.R1: the values you found on paper
answer_B4_R1a = ...  # a) a whole number
answer_B4_R1b = ...  # b) True or False
answer_B4_R1c = ...  # c) a whole number
answer_B4_R1d = ...  # d) the letter of your choice, e.g. "E"
answer_B4_R1e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B4_R1a, answer_B4_R1b, answer_B4_R1c, answer_B4_R1d, answer_B4_R1e]):
    wb.check(f"B4.R1{letter}", answer)

**B4.R2 — Extraction de features ou fine-tuning : que change le nombre d'exemples ?**

In [ ]:
# B4.R2: the values you found on paper
answer_B4_R2a = ...  # a) the letter of your choice, e.g. "E"
answer_B4_R2b = ...  # b) True or False
answer_B4_R2c = ...  # c) the letter of your choice, e.g. "E"
answer_B4_R2d = ...  # d) True or False
answer_B4_R2e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_B4_R2a, answer_B4_R2b, answer_B4_R2c, answer_B4_R2d, answer_B4_R2e]):
    wb.check(f"B4.R2{letter}", answer)

**B4.R3 — Les k plus proches voisins et le choix de la distance**

In [ ]:
# B4.R3: the values you found on paper
answer_B4_R3a = ...  # a) a number, 2 decimals
answer_B4_R3b = ...  # b) a number, 2 decimals
answer_B4_R3c = ...  # c) a number, 2 decimals
answer_B4_R3d = ...  # d) True or False
answer_B4_R3e = ...  # e) a whole number

for letter, answer in zip("abcde", [answer_B4_R3a, answer_B4_R3b, answer_B4_R3c, answer_B4_R3d, answer_B4_R3e]):
    wb.check(f"B4.R3{letter}", answer)

**Ex B4.1 — Mémoire d'un modèle : paramètres, dtype et KV cache — tient-il sur ton ordinateur ?**

In [ ]:
# Ex B4.1: the values you found on paper
answer_B4_1a = ...  # a) a whole number
answer_B4_1b = ...  # b) a number, 1 decimal
answer_B4_1c = ...  # c) a number, 1 decimal
answer_B4_1d = ...  # d) a number, 1 decimal
answer_B4_1e = ...  # e) a number, 1 decimal
answer_B4_1f = ...  # f) a whole number
answer_B4_1g = ...  # g) a number, 1 decimal
answer_B4_1h = ...  # h) True or False
answer_B4_1i = ...  # i) a whole number
answer_B4_1j = ...  # j) a whole number
answer_B4_1k = ...  # k) True or False

for letter, answer in zip("abcdefghijk", [answer_B4_1a, answer_B4_1b, answer_B4_1c, answer_B4_1d, answer_B4_1e, answer_B4_1f, answer_B4_1g, answer_B4_1h, answer_B4_1i, answer_B4_1j, answer_B4_1k]):
    wb.check(f"B4.1{letter}", answer)

**Ex B4.2 — Recall@k et MRR pour cinq requêtes**

In [ ]:
# Ex B4.2: the values you found on paper
answer_B4_2a = ...  # a) a number, 3 decimals
answer_B4_2b = ...  # b) a number, 3 decimals
answer_B4_2c = ...  # c) a number, 3 decimals
answer_B4_2d = ...  # d) a number, 3 decimals
answer_B4_2e = ...  # e) a number, 3 decimals
answer_B4_2f = ...  # f) True or False
answer_B4_2g = ...  # g) a whole number
answer_B4_2h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_B4_2a, answer_B4_2b, answer_B4_2c, answer_B4_2d, answer_B4_2e, answer_B4_2f, answer_B4_2g, answer_B4_2h]):
    wb.check(f"B4.2{letter}", answer)

**Ex B4.3 — LoRA : paramètres entraînables d'une couche 896×896 de rang 8, puis de tout un modèle**

In [ ]:
# Ex B4.3: the values you found on paper
answer_B4_3a = ...  # a) a whole number
answer_B4_3b = ...  # b) a number, 2 decimals
answer_B4_3c = ...  # c) a number, 2 decimals
answer_B4_3d = ...  # d) a whole number
answer_B4_3e = ...  # e) a whole number
answer_B4_3f = ...  # f) a number, 4 decimals
answer_B4_3g = ...  # g) a whole number
answer_B4_3h = ...  # h) True or False
answer_B4_3i = ...  # i) a number, 1 decimal
answer_B4_3j = ...  # j) a whole number
answer_B4_3k = ...  # k) True or False

for letter, answer in zip("abcdefghijk", [answer_B4_3a, answer_B4_3b, answer_B4_3c, answer_B4_3d, answer_B4_3e, answer_B4_3f, answer_B4_3g, answer_B4_3h, answer_B4_3i, answer_B4_3j, answer_B4_3k]):
    wb.check(f"B4.3{letter}", answer)

**Ex B4.4 — Un chat template à la main : rôles, tokens spéciaux et prompt de génération**

In [ ]:
# Ex B4.4: the values you found on paper
answer_B4_4a = ...  # a) a whole number
answer_B4_4b = ...  # b) a whole number
answer_B4_4c = ...  # c) a whole number
answer_B4_4d = ...  # d) a whole number
answer_B4_4e = ...  # e) a whole number
answer_B4_4f = ...  # f) the letter of your choice, e.g. "E"
answer_B4_4g = ...  # g) True or False
answer_B4_4h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_B4_4a, answer_B4_4b, answer_B4_4c, answer_B4_4d, answer_B4_4e, answer_B4_4f, answer_B4_4g, answer_B4_4h]):
    wb.check(f"B4.4{letter}", answer)

**Ex B4.5 — BM25 à la main : trois documents, une requête de deux mots**

In [ ]:
# Ex B4.5: the values you found on paper
answer_B4_5a = ...  # a) a number, 4 decimals
answer_B4_5b = ...  # b) a number, 4 decimals
answer_B4_5c = ...  # c) a number, 4 decimals
answer_B4_5d = ...  # d) a number, 4 decimals
answer_B4_5e = ...  # e) a number, 4 decimals
answer_B4_5f = ...  # f) a list of document ids, best first
answer_B4_5g = ...  # g) True or False
answer_B4_5h = ...  # h) a number, 4 decimals
answer_B4_5i = ...  # i) True or False

for letter, answer in zip("abcdefghi", [answer_B4_5a, answer_B4_5b, answer_B4_5c, answer_B4_5d, answer_B4_5e, answer_B4_5f, answer_B4_5g, answer_B4_5h, answer_B4_5i]):
    wb.check(f"B4.5{letter}", answer)

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu dire si un modèle tient en mémoire, poids et KV cache compris, selon son dtype ?
2. Sais-tu appliquer à la main un chat template et compter les tokens d'un prompt ?
3. Sais-tu calculer BM25, recall@k et MRR à la main ?
4. Sais-tu compter les paramètres de LoRA et dériver les gradients de $\mathbf{A}$ et $\mathbf{B}$ ?

**Pour aller plus loin** : le cours LLM de Hugging Face et l'article de LoRA, cités dans la fiche. La suite : le bonus B5, les modèles génératifs d'images.

*Les exercices de code B4.12 à B4.30 (usage et prompting, recherche et RAG, fine-tuning, livraison) seront ajoutés à la prochaine session de génération ; relance alors `python tools/start_chapter.py B4` pour obtenir le notebook complet.*